In [1]:
import pandas as pd

In [42]:
path = r'data/KQTT 2024.xlsx'
file = pd.read_excel(path, sheet_name='Sheet1')

In [33]:
import unicodedata
import re

def remove_vietnamese_diacritics(text):
    if not isinstance(text, str):
        return text
    text = text.replace('đ', 'd').replace('Đ', 'D')
    text = unicodedata.normalize('NFD', text)
    return re.sub(r'[\u0300-\u036f]', '', text).lower()

# 1. Tạo cột không dấu và viết thường
file['nhasx_clean'] = file['lowered_nhasx'].apply(remove_vietnamese_diacritics)
# 1. Danh sách từ khóa cần tìm
keywords = ['imexpharm']
pattern = '|'.join(keywords)

is_imexpharm = file['lowered_nhasx'].str.contains(pattern, na=False)
not_agimexpharm = ~file['lowered_nhasx'].str.contains('agimexpharm', na=False)
# 3. Lọc DataFrame

result = file.loc[is_imexpharm & not_agimexpharm].assign(ticker='IMP')

In [52]:
import json
import re
import unicodedata
import pandas as pd

def remove_vietnamese_diacritics(text):
    if pd.isna(text) or text is None:
        return ""
    text = str(text).replace('đ', 'd').replace('Đ', 'D')
    text = unicodedata.normalize('NFD', text)
    text = re.sub(r'[\u0300-\u036f]', '', text)
    return unicodedata.normalize('NFC', text).strip().lower()

def build_regex_pattern(keywords):
    if not keywords:
        return None
    if isinstance(keywords, str):
        keywords = [keywords]
    escaped = [
        re.escape(remove_vietnamese_diacritics(k)) 
        for k in keywords 
        if k and str(k).strip()
    ]
    return '|'.join(escaped) if escaped else None

def filter_by_ticker_config(
    df: pd.DataFrame, 
    config, 
    col, 
    output_col: str = 'ticker', 
    return_unmatched: bool = False, 
    verbose: bool = True
):
    """
    Lọc DataFrame theo cấu hình Ticker/Từ khóa và gán nhãn vào cột tùy chọn.
    
    Parameters:
    - df: DataFrame gốc
    - config: dict / JSON string (1 config) hoặc list các dict / JSON string (nhiều configs)
    - col: Tên cột hoặc chỉ số cột cần tìm kiếm (mặc định: 17)
    - output_col: Tên cột kết quả được tạo ra (mặc định: 'ticker')
    - return_unmatched: Trả về (df_matched, df_unmatched) nếu True
    - verbose: In báo cáo thống kê nếu True
    
    Returns:
    - pd.DataFrame (hoặc tuple (matched_df, unmatched_df))
    """
    if isinstance(config, str):
        config = json.loads(config)
    
    # 1. Chuẩn hóa cột dữ liệu 1 lần duy nhất để tối ưu hiệu năng
    clean_series = df[col].apply(remove_vietnamese_diacritics)
    
    configs_list = config if isinstance(config, list) else [config]
    matched_dfs = []
    
    for cfg in configs_list:
        # Lấy giá trị nhãn: ưu tiên lấy theo tên output_col, nếu không có thì lấy 'ticker' hoặc 'label'
        label_val = cfg.get(output_col, cfg.get('ticker', cfg.get('label', '')))
        
        include_kws = cfg.get('include_keyword', cfg.get('include_keywords', []))
        exclude_kws = cfg.get('exclude_keyword', cfg.get('exclude_keywords', []))
        
        include_pat = build_regex_pattern(include_kws)
        exclude_pat = build_regex_pattern(exclude_kws)
        
        if not include_pat:
            continue
            
        cond = clean_series.str.contains(include_pat, na=False)
        if exclude_pat:
            cond = cond & (~clean_series.str.contains(exclude_pat, na=False))
            
        res = df.loc[cond].copy()
        res[output_col] = label_val  # Gán giá trị vào cột output_col tùy chọn
        matched_dfs.append(res)
        
    result_df = pd.concat(matched_dfs, ignore_index=False) if matched_dfs else pd.DataFrame()
    
    # Tính toán các dòng chưa match
    matched_idx = result_df.index.unique() if not result_df.empty else pd.Index([])
    unmatched_df = df.loc[~df.index.isin(matched_idx)].copy()
    
    if verbose:
        total = len(df)
        n_matched = len(matched_idx)
        n_unmatched = len(unmatched_df)
        print(f"=== BÁO CÁO MATCH DỮ LIỆU ===")
        print(f"Cột gán nhãn:       '{output_col}'")
        print(f"Tổng số dòng:       {total:,}")
        print(f"Đã match (gán nhãn): {n_matched:,} ({n_matched/total*100:.2f}%)")
        print(f"Chưa match:         {n_unmatched:,} ({n_unmatched/total*100:.2f}%)")
        if not result_df.empty:
            print(f"\nChi tiết theo '{output_col}':")
            print(result_df[output_col].value_counts().to_string())
        print("=" * 30)

    if return_unmatched:
        return result_df, unmatched_df
    return result_df

In [44]:
file.head()

,loai_thau,ma_tinh,ten_tinh,ten_don_vi,ma_cskcb,ten_cskcb,ma,ma_gy,ten,hoatchat,...,tieuchuan,nhomthau,loai,sttpheduyet,hieuluc,congbo,ht_thau,tungay_hd,denngay_hd,created_date
0,3.thau_rieng_le,83,Tỉnh Bến Tre,Tyt Quới Điền,83712,Tyt Quới Điền,40.998,40.998,Acetylcystein,Acetylcystein,...,NaN,N4,Tân dược,NaN,1.0,2024-08-16 00:00:00,5.0,2024-08-16 00:00:00,2025-02-11 00:00:00,2025-04-10 00:34:57
1,3.thau_rieng_le,35,Tỉnh Hà Nam,Tyt Nhân Khang,35136,Tyt Nhân Khang,40.81,40.81,Clorpheniramin,Chlorpheniramin (hydrogen maleat),...,NaN,N4,Tân dược,NaN,1.0,2024-03-04 00:00:00,1.0,2024-03-04 00:00:00,2025-03-04 00:00:00,2025-04-10 00:34:57
2,3.thau_rieng_le,31,Thành phố Hải Phòng,Bv Hữu Nghị Việt Tiệp,31153,Bv Hữu Nghị Việt Tiệp,40.805.2,40.805.2,Mixtard 30,"Insulin người trộn, hỗn hợp",...,NaN,N1,Tân dược,NaN,1.0,2024-11-25 00:00:00,1.0,2024-11-25 00:00:00,2025-11-25 00:00:00,2025-04-11 00:33:35
3,3.thau_rieng_le,82,Tỉnh Tiền Giang,Tyt Phú An,82144,Tyt Phú An,40.684,40.684,Sucralfate,Sucralfat,...,NaN,N4,Tân dược,NaN,1.0,2024-12-13 00:00:00,1.0,2024-12-13 00:00:00,2026-12-13 00:00:00,2025-04-11 00:33:35
4,3.thau_rieng_le,68,Tỉnh Lâm Đồng,Tyt Gia Lâm,68526,Tyt Gia Lâm,40.524,40.524,Ramipril DWP 5mg,Ramipril,...,NaN,N4,Tân dược,NaN,1.0,2024-12-20 00:00:00,1.0,2024-12-20 00:00:00,2025-12-19 00:00:00,2025-04-11 00:33:35


In [57]:
# 1. Danh sách cấu hình các mã cổ phiếu ngành Dược (có thể thêm bớt tùy ý)
configs = [
    {
        "ticker": "IMP",
        "include_keyword": ["imexpharm", "dp imexpharm", "cpdp imexpharm"],
        "exclude_keyword": ["agimexpharm"]
    },
    # {
    #     "ticker": "DHG",
    #     "include_keyword": ["duoc hau giang", "dhg"],
    #     "exclude_keyword": []
    # },
    # {
    #     "ticker": "TRA",
    #     "include_keyword": ["traphaco"],
    #     "exclude_keyword": []
    # },
    # {
    #     "ticker": "DBD",
    #     "include_keyword": ["bidiphar", "duoc trang thiet bi y te binh dinh"],
    #     "exclude_keyword": []
    # },
    # {
    #     "ticker": "DMC",
    #     "include_keyword": ["domesco"],
    #     "exclude_keyword": []
    # }
]

# 2. Lọc dữ liệu trên DataFrame gốc (cột 17 là cột nhà sản xuất)
result_df = filter_by_ticker_config(file, configs, col='nhasx', output_col='ma_ck')
# Cột mới tạo ra sẽ có tên là 'ma_ck' thay vì 'ticker'
result_df.head()


=== BÁO CÁO MATCH DỮ LIỆU ===
Cột gán nhãn:       'ma_ck'
Tổng số dòng:       273,697
Đã match (gán nhãn): 3,470 (1.27%)
Chưa match:         270,227 (98.73%)

Chi tiết theo 'ma_ck':
ma_ck
IMP    3470


,loai_thau,ma_tinh,ten_tinh,ten_don_vi,ma_cskcb,ten_cskcb,ma,ma_gy,ten,hoatchat,...,nhomthau,loai,sttpheduyet,hieuluc,congbo,ht_thau,tungay_hd,denngay_hd,created_date,ma_ck
20,3.thau_rieng_le,01,Thành phố Hà Nội,Bv Bnđ Tw,01939,Bv Bnđ Tw,40.249,40.249,Colistin 1 MIU,Colistin*,...,N2,Tân dược,NaN,1.0,2024-01-16 00:00:00,1.0,2024-01-16 00:00:00,2025-01-16 00:00:00,2025-04-10 00:34:57,IMP
86,3.thau_rieng_le,11,Tỉnh Điện Biên,Tyt Sính Phình,11067,Tyt Sính Phình,40.158,40.158,"Nerusyn 1,5g",Ampicilin + sulbactam,...,N2,Tân dược,NaN,1.0,2024-01-16 00:00:00,1.0,2024-01-16 00:00:00,2024-12-31 00:00:00,2025-04-10 00:34:57,IMP
91,3.thau_rieng_le,11,Tỉnh Điện Biên,Tyt Thanh Yên,11020,Tyt Thanh Yên,40.190,40.190,Oxacillin 1g,Oxacilin,...,N2,Tân dược,NaN,1.0,2024-02-05 00:00:00,1.0,2024-02-05 00:00:00,2025-02-05 00:00:00,2025-04-11 00:33:35,IMP
274,3.thau_rieng_le,22,Tỉnh Quảng Ninh,Tyt-Công Ty Than Quang Hanh-Tkv,22146,Tyt-Công Ty Than Quang Hanh-Tkv,40.163,40.163,Opxil IMP 500 mg,Cefalexin,...,N1,Tân dược,NaN,1.0,2024-05-26 00:00:00,3.0,2024-05-26 00:00:00,2024-12-31 00:00:00,2025-04-10 00:34:57,IMP
306,3.thau_rieng_le,68,Tỉnh Lâm Đồng,Tyt Sơn Điền,68584,Tyt Sơn Điền,40.161,40.161,Imeclor 125,Cefaclor,...,N2,Tân dược,NaN,1.0,2024-09-10 00:00:00,1.0,2024-09-10 00:00:00,2025-09-10 00:00:00,2025-04-11 00:33:35,IMP


In [58]:
result_df.to_excel('outputs/IMP.xlsx', index=False)